In [2]:
import pandas as pd
import numpy as np

In [4]:
# 1. Chargement des données brutes avec un chemin relatif portable
DATA_RAW_PATH = "../data/raw/salaries.csv"
df = pd.read_csv(DATA_RAW_PATH)
# 2. Relevé chiffré des dimensions initiales
initial_rows, initial_cols = df.shape
print(f"Jeu de données initial : {initial_rows:,} observations et {initial_cols} variables.")

# 3. Aperçu des premières lignes
df.head()

Jeu de données initial : 93,597 observations et 11 variables.


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2025,MI,FT,Research Scientist,208000,USD,208000,US,0,US,M
1,2025,MI,FT,Research Scientist,147000,USD,147000,US,0,US,M
2,2025,SE,FT,Research Scientist,173000,USD,173000,US,0,US,M
3,2025,SE,FT,Research Scientist,117000,USD,117000,US,0,US,M
4,2025,MI,FT,AI Engineer,100000,USD,100000,US,100,US,M


In [5]:
# 1. Structure générale et types détectés par Pandas
print("--- Types de données et informations générales ---")
df.info()

# 2. Audit chiffré des valeurs manquantes par colonne
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100

missing_df = pd.DataFrame({
    'Manquants': missing_count,
    'Pourcentage_%': missing_percent.round(2)
})

print("\n--- Bilan des valeurs manquantes ---")
# Afficher les colonnes ayant au moins une valeur manquante
missing_cols = missing_df[missing_df['Manquants'] > 0]
if missing_cols.empty:
    print("Aucune valeur manquante détectée : le jeu de données est complet.")
else:
    print(missing_cols)

--- Types de données et informations générales ---
<class 'pandas.DataFrame'>
RangeIndex: 93597 entries, 0 to 93596
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   work_year           93597 non-null  int64
 1   experience_level    93597 non-null  str  
 2   employment_type     93597 non-null  str  
 3   job_title           93597 non-null  str  
 4   salary              93597 non-null  int64
 5   salary_currency     93597 non-null  str  
 6   salary_in_usd       93597 non-null  int64
 7   employee_residence  93597 non-null  str  
 8   remote_ratio        93597 non-null  int64
 9   company_location    93597 non-null  str  
 10  company_size        93597 non-null  str  
dtypes: int64(4), str(7)
memory usage: 7.9 MB

--- Bilan des valeurs manquantes ---
Aucune valeur manquante détectée : le jeu de données est complet.


In [ ]:
# 1. Relevé chiffré des doublons avant action
n_duplicates = df.duplicated().sum()
pct_duplicates = (n_duplicates / len(df)) * 100

print(f"Doublons stricts détectés : {n_duplicates:,} lignes ({pct_duplicates:.2f} %)")

# 2. Suppression des doublons
df_cleaned = df.drop_duplicates().copy()

# 3. Validation de l'impact
print(f"Dimensions après déduplication : {df_cleaned.shape[0]:,} observations restantes.")

In [6]:
# 1. Relevé chiffré des doublons avant action
n_duplicates = df.duplicated().sum()
pct_duplicates = (n_duplicates / len(df)) * 100

print(f"Doublons stricts détectés : {n_duplicates:,} lignes ({pct_duplicates:.2f} %)")

# 2. Suppression des doublons
df_cleaned = df.drop_duplicates().copy()

# 3. Validation de l'impact
print(f"Dimensions après déduplication : {df_cleaned.shape[0]:,} observations restantes.")

Doublons stricts détectés : 46,960 lignes (50.17 %)
Dimensions après déduplication : 46,637 observations restantes.


In [7]:
# Identifier le nom exact de la colonne salaire (souvent 'salary_in_usd')
target_col = 'salary_in_usd' if 'salary_in_usd' in df_cleaned.columns else 'salary'

# 1. Calcul des quartiles et de l'IQR (Interquartile Range)
Q1 = df_cleaned[target_col].quantile(0.25)
Q3 = df_cleaned[target_col].quantile(0.75)
IQR = Q3 - Q1

# Bornes de Tukey
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 2. Comptage chiffré des valeurs aberrantes
outliers = df_cleaned[(df_cleaned[target_col] < lower_bound) | (df_cleaned[target_col] > upper_bound)]
n_outliers = len(outliers)
pct_outliers = (n_outliers / len(df_cleaned)) * 100

print(f"--- Détection des valeurs aberrantes ({target_col}) ---")
print(f"Q1 (25 %) : {Q1:,.0f} $ | Médiane : {df_cleaned[target_col].median():,.0f} $ | Q3 (75 %) : {Q3:,.0f} $")
print(f"Borne inférieure : {lower_bound:,.0f} $ | Borne supérieure : {upper_bound:,.0f} $")
print(f"Nombre d'outliers identifiés : {n_outliers} ({pct_outliers:.2f} %)")

--- Détection des valeurs aberrantes (salary_in_usd) ---
Q1 (25 %) : 96,400 $ | Médiane : 139,100 $ | Q3 (75 %) : 190,500 $
Borne inférieure : -44,750 $ | Borne supérieure : 331,650 $
Nombre d'outliers identifiés : 1114 (2.39 %)


In [8]:
# 1. Définition du chemin cible relatif
PROCESSED_DATA_PATH = "../data/processed/clean_salaries.csv"

# 2. Sauvegarde du DataFrame nettoyé
# index=False évite d'enregistrer une colonne d'index inutile
df_cleaned.to_csv(PROCESSED_DATA_PATH, index=False)

print(f"✅ Données nettoyées enregistrées avec succès : {df_cleaned.shape[0]:,} lignes dans '{PROCESSED_DATA_PATH}'.")


✅ Données nettoyées enregistrées avec succès : 46,637 lignes dans '../data/processed/clean_salaries.csv'.
